In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import wandb
from hydra import initialize, initialize_config_module, initialize_config_dir, compose

from datetime import datetime

from machine_intelligence.utilities.data_handling import load_parquet_table

from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule

from machine_intelligence.utilities.viz_setup import figure_setup, VariableNames


In [ ]:
aligned_data_path = Path("df_all_aligned.parquet")

In [ ]:
df_aligned = load_parquet_table(aligned_data_path)

In [ ]:
df_aligned.head()

Gute samples: 11

In [ ]:
df_aligned_sample = df_aligned.groupby(by="patientunitstayid").get_group(df_aligned["patientunitstayid"].unique()[11])

In [ ]:

df_aligned_sample["time_offset_hours"] = df_aligned_sample["time_offset"].dt.total_seconds() / 3600


In [ ]:

df_aligned_sample = df_aligned_sample.dropna(axis=1, how="all")

lab_names = VariableNames.get_lab_related_names()
lab_names_original = list(lab_names.keys())
lab_names_pretty = list(lab_names.values())

vital_names = VariableNames.get_vital_sign_names()
vital_names_original = list(vital_names.keys())
vital_names_pretty = list(vital_names.values())

df_aligned_sample_present = VariableNames.get_pretty_dataframe(df_aligned_sample)

vital_names_pretty_present = [name for name in vital_names_pretty if name in df_aligned_sample_present.columns]
lab_names_pretty_present = [name for name in lab_names_pretty if name in df_aligned_sample_present.columns]

df_aligned_sample.head()

In [ ]:

import palettable


def plot_sparse_with_random_markers(ax, x, y, marker_list):
    for col in y.columns:
        # Randomly selecting marker positions
        marker_scatter = np.random.choice(marker_list)

        mask = np.random.choice([True, False], size=len(x), p=[0.3, 0.7])  # Adjust p for marker frequency
        # ax.plot(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7)
        jittered_x = x[mask] + np.random.normal(0, 0.3, size=len(x[mask]))  # Adjust 0.1 for more or less jitter
        ax.scatter(jittered_x, y[col][mask], marker=marker_scatter, linestyle='', label=col, s=30,
                   alpha=0.7,

                   # hatch=np.random.randint(0,6) * "|" 
                   )
        # ax.scatter(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7, s=10)


# Generate the figure and axes
fig_control, axs_control, _, font_annots = figure_setup(2, 1, height_cm=20,
                                                        palette=palettable.matplotlib.Plasma_11.mpl_colors[:10],
                                                        share_x=True)

line_styles = ["-", "--", "-.", ":", "-", "--", "-.", ":", "-", "--", "-.", ":", ]

# Plot dense ts variables with different line styles
for i, col in enumerate(vital_names_pretty_present):
    axs_control[0].plot(df_aligned_sample_present["time_offset_hours"],
                        df_aligned_sample_present[col],
                        linestyle=line_styles[i % len(line_styles)],
                        label=col)

# sparse ts variables with random markers
plot_sparse_with_random_markers(axs_control[1],
                                df_aligned_sample_present["time_offset_hours"],
                                df_aligned_sample_present[lab_names_pretty_present].replace(0.0, pd.NA),
                                marker_list=['x', 'o', 's', 'D', '*', "h", "+", "p", "2", "8"])

for ax in axs_control:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 6))
    ax.set_xlim(left=0, right=df_aligned_sample_present["time_offset_hours"].max())

#axs_control[0].set_ylim(0, 1)


axs_control[0].set_ylabel("Dense Time Series Features")
axs_control[1].set_ylabel("Sparse Time Series Features")

axs_control[1].set_xlabel("Time [h]")

font_annots.set_size(6)  # Set the desired font size
# Customizing legends
axs_control[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs_control[1].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.5),
    ncol=4,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()


In [ ]:


cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:

if save_as_svg := True:
    fig_control.savefig(plot_output_dir / Path("admission_data_illustration.svg"), format="svg")


In [ ]:

def plot_sparse_with_random_markers(ax, x, y, marker_list):
    for col in y.columns:
        # Randomly selecting marker positions
        marker_scatter = np.random.choice(marker_list)

        mask = np.random.choice([True, False], size=len(x), p=[0.3, 0.7])  # Adjust p for marker frequency
        # ax.plot(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7)
        jittered_x = x[mask] + np.random.normal(0, 0.3, size=len(x[mask]))  # Adjust 0.1 for more or less jitter
        ax.scatter(jittered_x, y[col][mask], marker=marker_scatter, linestyle='', label=col, s=50,
                   alpha=0.8,
                   zorder=100
                   )
        # ax.scatter(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7, s=10)


# Generate the figure and axes
fig_control, axs_control, _, font_annots = figure_setup(1, 1, height_cm=14, width_cm=14,
                                                        palette=palettable.matplotlib.Plasma_11.mpl_colors[:10],
                                                        share_x=True)

line_styles = ["-", "--", "-.", ":", "-", "--", "-.", ":", "-", "--", "-.", ":", ]

# Plot dense ts variables with different line styles
for i, col in enumerate(vital_names_pretty_present):
    axs_control.plot(df_aligned_sample_present["time_offset_hours"],
                     df_aligned_sample_present[col],
                     linestyle=line_styles[i % len(line_styles)],
                     label=col)

# sparse ts variables with random markers
plot_sparse_with_random_markers(axs_control,
                                df_aligned_sample_present["time_offset_hours"],
                                df_aligned_sample_present[lab_names_pretty_present].replace(0.0, pd.NA),
                                marker_list=['x', 'o', 's', 'D', '*', "h", "+", "p", "2", "8"])

# for ax in axs_control:
axs_control.xaxis.set_major_locator(plt.MultipleLocator(1 * 6))
axs_control.set_xlim(left=0, right=df_aligned_sample_present["time_offset_hours"].max())

axs_control.set_ylim(0, 200)

axs_control.set_ylabel("Sparse &  Dense Time Series Features")
# axs_control[1].set_ylabel("Sparse Time Series Features")

axs_control.set_xlabel("Time [h]")

font_annots.set_size(6)  # Set the desired font size
# Customizing legends
# axs_control.legend(
#     loc="upper center",
#     bbox_to_anchor=(0.5, -0.3),
#     ncol=3,
#     frameon=False,
#     fancybox=True,
#     shadow=True,
#     prop=font_annots,
# )
# 
axs_control.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()



In [ ]:
fig_control.savefig(plot_output_dir / Path("admission_data_illustration.svg"), format="svg")


In [ ]:
from eicu_code_dhm.workspace_dani.data_preprocessing.utils.data_quality import (
    get_parameter_coverage_ratio_df,
    categorical_feature_frequency,
)



In [ ]:
df_ds_grouped = df_aligned.groupby(by="patientunitstayid")

df_variable_densities = get_parameter_coverage_ratio_df(df_grouped=df_ds_grouped, parameters=df_aligned.columns)

df_variable_densities.describe()


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

# parameters_to_plot = ["sao2", "heartrate", "temperature", "noninvasivesystolic", "st1", "labname_-monos",
#                       "labname_creatinine", "labname_MCHC"]


parameters_to_plot = ["Heart Rate", "Oxygen Saturation (SaO2)", "Respiration", "Noninvasive Systolic",
                      "Bedside Glucose", "White Blood Cell Count", "Lactate", "Creatinine", "Blood Urea Nitrogen (BUN)"]

df_variable_densities = VariableNames.get_pretty_dataframe(df_variable_densities)
df_variable_densities.head()

In [ ]:
df_variable_densities.describe()

In [ ]:



fig_dens, axs_dens, _, _ = figure_setup(1, 1, width_cm=10, height_cm=10)

sns.boxenplot(df_variable_densities[parameters_to_plot], ax=axs_dens,
              # sns.boxenplot(df_variable_densities, ax=axs_dens,
              palette=palettable.matplotlib.Plasma_12.mpl_colors[3:]
              )

axs_dens.set_ylabel("Data Density $f_{grid}=5$ min")
plt.xticks(rotation=30, ha="right")
# axs_dens.set_xticklabels(
#     ["SaO2", "Heart Rate", "Temperature", "Non-invasive Systolic", "St1", "Monocytes", "Creatinine", "MCHC"])

plt.tight_layout()
if save_as_svg := True:
    fig_dens.savefig(plot_output_dir / Path("variable_densities.svg"), format="svg")

